# 🚀 Модуль 4. Урок 4. Автоматизация очистки: построение воспроизводимых конвейеров

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_04_data_cleaning/lesson_04_automated_cleaning_pipelines.ipynb)

## 📌 Бизнес-контекст и методология
Аналитик не очищает данные руками в блокноте. Он оформляет логику в пайплайн, готовый к запуску в Airflow или dbt.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import pandas as pd

def clean_orders_pipeline(raw_orders):
    df = raw_orders.copy()
    df['order_date'] = pd.to_datetime(df['order_date'])
    df = df.dropna(subset=['order_id', 'customer_id'])
    df = df[df['status'] != 'Cancelled']
    return df

df_raw = pd.read_csv(orders_csv)
df_clean = clean_orders_pipeline(df_raw)
print(f"До очистки: {len(df_raw)} строк, после: {len(df_clean)} строк")


До очистки: 2282 строк, после: 2162 строк


### 🎯 Практическое задание 1: Добавление валидации пайплайна
Добавьте проверку `assert not df_clean.empty` и убедитесь, что датасет валиден.

In [3]:
### ВАШ КОД ЗДЕСЬ
assert not df_clean.empty, "Датафрейм не должен быть пустым"
assert 'order_id' in df_clean.columns, "Колонка order_id обязательна"
print("✅ Тест успешно пройден: пайплайн очистки готов к продакшену!")


✅ Тест успешно пройден: пайплайн очистки готов к продакшену!


### 💡 Архитектура Data Quality
В 2027 году используются инструменты Great Expectations и Soda Core для проверки контрактов данных (Data Contracts).